In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
from handcalcs import *
from handcalcs import render
import handcalcs

In [3]:
def trigger (sig , thresh= None, transition= "rise", marginI = 20):
    arr=[0]
    if thresh is None: 
        thresh = (sig.max()+sig.min())/2
    for i in range(1, len(sig)):
        if transition =="rise":
            if sig[i-1] <thresh and sig[i] >= thresh and i > arr[-1]+marginI:
                arr.append(i)
        elif sig[i-1] >thresh and sig[i] <= thresh and i > arr[-1]+marginI:
            arr.append(i)
    return arr[1:]


In [4]:
# Initialization 
bl= '#1520c0' # blue 
rd= '#C62828' # red
fig_counter=1

In [5]:
# CONFIGURATIONS
#help(handcalcs.set_option)
handcalcs.set_option("latex_block_start", "$")
handcalcs.set_option("latex_block_end", "$")
handcalcs.set_option("math_environment_end", "aligned")
handcalcs.set_option("use_scientific_notation",True)

In [6]:
from figModel import *

# Main HV architecture of the LLC converter

<br><br><br><br><br>
<img src="figs/une2.png" >

## Table of contents

## Primary mosfet

For this LLC DC–DC converter, we can choose between a full‑bridge or a half‑bridge configuration on the primary side. Here is a comparison of the two approaches:

| Feature                              | Half Bridge LLC       | Full Bridge LLC       |
| ------------------------------------ | --------------------- | --------------------- |
| **Number of switches**               | 2                     | 4                     |
| **Voltage applied to resonant tank** | $V_{in}/2$            | $V_{in}$              |
| **Power capability(*)**                 | Medium                | High                  |
| **Transformer turns ratio**          | Higher                | Lower                 |
| **Conduction losses(*)**                | Higher (more current) | Lower                 |
| **Cost**                             | Lower                 | Higher                |
| **Control complexity**               | Simpler               | Slightly more complex |


(*) : In a half‑bridge topology, the applied voltage is half that of a full bridge, so delivering the same power requires roughly twice the current. This higher current increases copper losses, which in turn restricts the maximum power capability.

In [7]:
fig_counter=One_figure_with_captions(fig_counter, img1="Primary.png"
                                     , caption1="Primary mosfet configuration: Left: Full bridge, Right: Half bridge",
                                     first_part_path="figs/", 
                                     width=12)

We will select the half‑bridge topology because it offers lower cost and simpler control. This choice is fully compatible with our 1.2 kW power target.

## Resonant capacitor

For the half‑bridge configuration, we have two options: using one bloc of resonant capacitor or dividing it into two separate blocs of capacitors (see the illustration below).

In [8]:
fig_counter=One_figure_with_captions(fig_counter, img1="PrimaryTank.png"
                                     , caption1="LLC tank configuration: Left: One resonant capacitor, Right: Splited resonant capacitor",
                                     first_part_path="figs/", 
                                     width=19)

Below is a comparison of the two configurations.

Both designs use identical values for Lm, Lr, and C, and the same total number of capacitors.
The only distinction is that:
* In the first simulation, the capacitors are split into two blocks, with one connected to HV+ and the other to HV-. 
* In the second simulation, all capacitors are combined into a single block connected to HV-.

In [9]:
fig_counter=One_figure_with_captions(fig_counter, img1="Comparaison_Res_Capas_Archi2.png"
                                     , caption1="LLC tank configuration: Left: One resonant capacitor, Right: Splited resonant capacitor",
                                     first_part_path="./simulation/", 
                                     width=19)

From a power-electronics standpoint, the two configurations behave very similarly.
<br>This is evident when comparing their small-signal models:

the capacitor voltage (green curve) and the current (red curve) show almost identical waveforms in both implementations.

Therefore, we will choose the simpler approach: grouping all the capacitors into a single block, primarily to optimize the PCB layout and reduce area usage.

But for **EMI** constraints, we must keep in mind that the symmetric solution, which consists of splitting the capacitors into two blocks, is better, and the chosen solution creates an **asymmetric** path that can worsen EMI disturbances.

## Secondary mosfets

We can choose between using **diodes** or **MOSFETs** in the secondary stage.

At 1.2 kW and 48 V output, the output current is around 25 A. 
<br>Depending on the configuration, but if we assume this same current flows continuously through a MOSFET with a voltage drop of 0.2 V and through a diode with a drop of 0.8 V, the losses would be 5 W in the MOSFET and 20 W in the diode.

Diodes are still used as secondary rectifiers in some modern 11 kW EV chargers, but the key difference is the output voltage. At around 400 V output, a 0.2 V MOSFET drop or a 1.2 V diode drop has minimal impact on efficiency. However, at low‑voltage, high‑current outputs such as 12 V, this voltage drop becomes significant, making MOSFETs the better choice.

In conclusion, rectifier diodes are low‑cost, require no control, and simplify the layout, but they offer the poorest efficiency and create major challenges in thermal management.
* For this reason, we will use MOSFETs in the output stage.

However, we still have three possible secondary‑side configurations, as shown in the figure below:

*   A full‑bridge rectifier
*   A half‑bridge with a center‑tapped transformer and MOSFETs, where the winding is pulled down to ground
*   A half‑bridge with a center‑tapped transformer and MOSFETs, where the winding is pulled up to the positive rail (V+)


In [10]:
fig_counter=One_figure_with_captions(fig_counter, img1="SecondaryMos.png"
                                     , caption1="Secondary mosfet configuration:<br>(a): Full bridge, (b): Half bridge & source to ground, (c): Half bridge & drain to ground",
                                     first_part_path="figs/", 
                                     width=19)

Below a comparaison of those tree solutions:

| Feature                               | **Full-Bridge Rectifier**                                      | **Center-Tapped (Pull-Down MOSFETs)**                     | **Center-Tapped (Pull-Up MOSFETs)**                      |
| ------------------------------------- | -------------------------------------------------------------- | --------------------------------------------------------- | -------------------------------------------------------- |
| **Secondary topology**                | Single secondary winding with 4 rectifiers (diodes or MOSFETs) | Center-tapped secondary with 2 MOSFETs pulling to **GND** | Center-tapped secondary with 2 MOSFETs pulling to **V+** |
| **Number of rectifiers**              | 4 devices                                                      | 2 devices                                                 | 2 devices                                                |
| **Transformer requirement**           | Single secondary winding                                       | Center-tapped secondary winding                           | Center-tapped secondary winding                          |
| **Transformer copper usage**          | Better utilization                                             | Slightly worse (two half windings)                        | Slightly worse (two half windings)                       |
| **Current in secondary winding**      | Current flows through full winding                             | Current flows through half winding at a time              | Current flows through half winding at a time             |
| **Voltage stress on rectifiers**      | Lower                                                          | Higher                                                    | Higher                                                   |
| **Conduction path**                   | Two devices conduct each cycle                                 | One MOSFET conducts each half-cycle                       | One MOSFET conducts each half-cycle                      |
| **Conduction losses**                 | Higher (two drops in series)                                   | Lower                                                     | Lower                                                    |
| **RMS current per device**            | Lower per device                                               | Higher per device                                         | Higher per device                                        |
| **Gate drive complexity (MOSFET SR)** | Higher (4 devices to control)                                  | Lower  (ref to ground)                                                   | Lower  (but require insulation)                                                  |
| **PCB layout complexity**             | Higher                                                         | Moderate                                                  | More Simple                                                 |
| **Efficiency at low output voltage**  | Lower                                                          | Higher                                                    | Higher                                                   |



For these reasons, we will select the **middle option**, where the MOSFET source is connected to V– (ground).<br>
This configuration allows the gate driver to be ground‑referenced, making it fully compatible with our existing power‑supply architecture, as detailed in below in this document.

## Ground connection

For safety reasons, the primary side will be isolated, but for the output side we have two possible options:

Keep the output floating (isolated)
Reference the output to the system ground (OUT‑ connected to ground)
For example, in Tesla’s 400 V/48 V DC‑DC converters, OUT‑ is grounded.

Below is a comparison table of these two solutions:

| Feature                    | **Floating Output (Isolated)**                           | **Grounded Output (OUT- tied to GND)**                            |
| -------------------------- | -------------------------------------------------------- | ----------------------------------------------------------------- |
| **Isolation level**        | Full galvanic isolation maintained                       | Output referenced to ground |
| **Safety**                 | Higher protection from ground faults                     | Depends on system grounding scheme                        |
| **Common-mode noise**      | Usually higher                                           | Lower EMI and better noise control                                |
| **Measurement simplicity** | Harder (no reference)                                    | Easier (clear ground reference)                                   |
| **System integration**     | Flexible for different loads                             | Simpler integration with grounded systems                         |
| **Fault behavior**         | Floating nodes may drift                                 | Fault currents have defined return path                           |
| **EMI performance**        | Worse common-mode emissions                              | Often improved EMI behavior                                       |
| **PCB layout**   | More complexe |Simple output zone                         |


We will adopt the **grounded output** to simplify the PCB layout and improve EMI performance.

## The main full topology of the LLC DC‑DC

Below is the main topology of this LLC DC‑DC, including all the points discussed above.

In [11]:
fig_counter=One_figure_with_captions(fig_counter, img1="Full.png"
                                     , caption1="The final architecture of the 1200W LLC DCDC converter",
                                     first_part_path="figs/", 
                                     width=17)

## Auxiliary power supplies

 We selected the following topology for the auxiliary power supplies:   

<u>**Primary zone**</u>

*   Flyback: A flyback converter generates 12V from the 400V input. Its secondary ground is tied to HV-, and all primary‑side grounds are intended to be connected to HV- (HVN). The 12V output also supplies the low‑side driver, while the high‑side driver uses a bootstrap circuit.
*   A non‑isolated buck converter steps the 12V from the flyback down to 3.3V to power the MCU and all primary‑side signal components. 

<u>**Secondary zone**</u>
*   A non‑isolated buck converter steps the 48V main secondary voltage down to 12V to power the synchronous secondary drivers.
*   An LDO then converts the 12V to 3.3V for the isolated voltage amplifier on the secondary side.



In [12]:
fig_counter=One_figure_with_captions(fig_counter, img1="Power_supp2.png"
                                     , caption1="The topology for the auxiliary power supplies",
                                     first_part_path="figs/", 
                                     width=19)

## Primary and secondary drivers

  The primary‑side driver circuit is configured as follows:

*   **Low‑side VDDA** is powered directly from the flyback's 12V output.
*   **High‑side VDDB** is supplied by a bootstrap network that includes a current‑limiting resistor in series with a high‑voltage fast‑recovery diode.


In [13]:

fig_counter=One_figure_with_captions(fig_counter, img1="DriversPrim.png"
                                     , caption1="The primary side driver circuitry",
                                     first_part_path="figs/", 
                                     width=19)

As the secondary MOSFET sources are referenced to the secondary ground, the secondary synchronous drivers are supplied directly by the **12V output** of the buck converter.

## Architecture of sensors


The figure below show the architecture of the main seonsors: 

**Current senosrs**

* Input current: Isolated hall current sensor to measure the input current, powered by the 3.3v primary auxilary power supplay, and referensed to the HVN. 
* Output current: Isolated hall current sensor to measure the output current, powered by the 3.3v primary auxilary power supplay, and referensed to the HVN.

**Voltage sensors**

* Input voltage: Since the MUC is refereced to the HVN, this sensor is a simple voltage diveder between HVP and HVN. 
* Output voltage: A voltage divider flowed by an isolated amplifier. 

**Temperature sensors**

* Two CTN are used to meaure the PCB temperatures; one close to the primary mosfet region, the second is closed to the secondary mofset region. Each CTN is a low side thermal resistor in serial with a pull up precision resistor, and in parallel with a decoupling capacitor. 

The figure below shows the architecture of the main sensors:

**Current sensors**

*   Input current: An isolated Hall current sensor is used to measure the input current. It is powered by the 3.3 V primary auxiliary power supply and referenced to the HVN.
*   Output current: An isolated Hall current sensor is used to measure the output current. It is powered by the 3.3 V primary auxiliary power supply and referenced to the HVN.

**Voltage sensors**

*   Input voltage: As the MCU is referenced to the HVN, this measurement is implemented using a simple voltage divider between HVP and HVN.
*   Output voltage: A voltage divider followed by an isolated amplifier is used.

**Temperature sensors**

*   Two CTNs are used to measure PCB temperatures: one located near the primary MOSFET region and the other near the secondary MOSFET region. Each CTN is configured as a low-side thermal resistor in series with a precision pull-up resistor and in parallel with a decoupling capacitor.



In [14]:

fig_counter=One_figure_with_captions(fig_counter, img1="Sensors.png"
                                     , caption1="The architecture of the main sensors ",
                                     first_part_path="figs/", 
                                     width=19)

The STM32F1 MCU also offers the possibility to measure the CPU temperature, so this temperature will be used as a second-range sensor.

## HW protection

For safety reasons, a hardware protection circuit will be used.

*   The HW protection is composed of four fast open-drain comparators.
*   The outputs are connected to a common node with a pull-up resistor.
*   The comparator outputs are used to clear a D flip-flop, while the preset is controlled by the MCU.
*   The D flip-flop is used to disable the primary driver.
*   The four comparators are triggered by input/output voltage and input/output current sensor signals.
*   The comparator reference voltages are generated using four simple potentiometers.
*   Therefore, the HW protection functions include input/output OCP (over current protection) and input/output OVP (over voltage protection).


In [15]:

fig_counter=One_figure_with_captions(fig_counter, img1="hwPotection.png"
                                     , caption1="The HW protection circuit: Primary/Secondary OVP/OVP ",
                                     first_part_path="figs/", 
                                     width=19)

## Communication

Since the MCU is referenced to HVN, and for safety reasons, we will use an HV-isolated SPI interface.

*   This circuit will be powered by 3.3 V on the MCU side, and by an external 3.3 V supply on the isolated output side.
*   From the external point of view, the DC-DC will provide 6 communication pins: 4 for SPI and 2 to power the isolated interface.
*   The DC-DC MCU will operate as an SPI slave, while the external management circuit will be the SPI master.
*   Communication will be detailed in the SW design. In general, the DC-DC must report measurements such as input/output voltage and current, and receive output targets such as voltage, current, and power. If communication is not established, the DC-DC must apply default settings: 48 V output voltage and 1200 W maximum power (outside temperature derating).


In [16]:


fig_counter=One_figure_with_captions(fig_counter, img1="comm.png"
                                     , caption1="HV insulation circuit for SPI communication",
                                     first_part_path="figs/", 
                                     width=12)